# Imports

In [2]:
import os
import time
import gc
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

In [3]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from transformers import (
    ViTModel, ViTImageProcessor,
    DeiTModel, DeiTImageProcessor,
    CLIPProcessor, CLIPVisionModel, CLIPTextModel,
    BertTokenizer, BertModel, 
    RobertaTokenizer, RobertaModel,
    GPT2Tokenizer, GPT2Model
)

# Configuration

In [4]:
CURRENT_DATASET = "Flickr8k"
DATA_DIR = os.path.join(os.getcwd(), 'TFE_Data')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print(f"Dataset: {CURRENT_DATASET}")

Device: cuda
Dataset: Flickr8k


# Data Loading

In [5]:
df_path = os.path.join(DATA_DIR, f'subset_df_{CURRENT_DATASET}.pkl')
subset_df = pd.read_pickle(df_path)

# Extract lists for processing
image_paths = subset_df['image_path'].tolist()

# For the baseline unimodal indexation, we only use the first caption of each image
captions = [caps[0] for caps in subset_df['captions']] 

print(f"✅ Successfully loaded {len(image_paths)} image paths and corresponding captions.")

FileNotFoundError: [Errno 2] No such file or directory: '/home/aysel/tfe/TFE_Data/subset_df_Flickr8k.pkl'

# Unimodal Models

## CBIR Vision

### Indexing: Embedding Models

In [ ]:
def get_resnet50_embeddings(paths, device):
    """Extracts RAW visual features using heavy CNN (ResNet50)."""
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights).to(device).eval()
    model.fc = nn.Identity() # Strip classification head
    transform = weights.transforms()
    
    embeddings = []
    with torch.no_grad():
        for path in tqdm(paths, desc="Extracting ResNet50"):
            img = Image.open(path).convert('RGB')
            tensor = transform(img).unsqueeze(0).to(device)
            embeddings.append(model(tensor).squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [ ]:
def get_mobilenet_v3_embeddings(paths, device):
    """Extracts RAW visual features using lightweight CNN (MobileNetV3)."""
    weights = models.MobileNet_V3_Small_Weights.DEFAULT
    model = models.mobilenet_v3_small(weights=weights).to(device).eval()
    model.classifier[3] = nn.Identity() # Strip classification head
    transform = weights.transforms()
    
    embeddings = []
    with torch.no_grad():
        for path in tqdm(paths, desc="Extracting MobileNetV3"):
            img = Image.open(path).convert('RGB')
            tensor = transform(img).unsqueeze(0).to(device)
            embeddings.append(model(tensor).squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [ ]:
def get_vit_embeddings(paths, device):
    """Extracts RAW visual features using standard Transformer (ViT)."""
    processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
    model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for path in tqdm(paths, desc="Extracting ViT"):
            img = Image.open(path).convert('RGB')
            inputs = processor(images=img, return_tensors="pt").to(device)
            # Pooler output represents the [CLS] token global representation
            embeddings.append(model(**inputs).pooler_output.squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [ ]:
def get_deit_embeddings(paths, device):
    """Extracts RAW visual features using data-efficient Transformer (DeiT)."""
    processor = DeiTImageProcessor.from_pretrained('facebook/deit-base-distilled-patch16-224')
    model = DeiTModel.from_pretrained('facebook/deit-base-distilled-patch16-224').to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for path in tqdm(paths, desc="Extracting DeiT"):
            img = Image.open(path).convert('RGB')
            inputs = processor(images=img, return_tensors="pt").to(device)
            # Extract the first token representation (similar to [CLS])
            embeddings.append(model(**inputs).last_hidden_state[:, 0, :].squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [ ]:
def get_clip_vision_embeddings(paths, device):
    """Extracts RAW visual features using natively multimodal encoder (CLIP Vision)."""
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for path in tqdm(paths, desc="Extracting CLIP Vision"):
            img = Image.open(path).convert('RGB')
            inputs = processor(images=img, return_tensors="pt").to(device)
            embeddings.append(model(**inputs).pooler_output.squeeze().cpu().numpy())
    return model, np.array(embeddings)

## T2T Text 

### Indexing: Embedding Models

In [ ]:
# Custom BiLSTM Architecture for the RNN Baseline
class BiLSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=300, hidden_dim=384): 
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        # Concatenate forward and backward final hidden states (384 * 2 = 768 dimensions)
        return torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)

In [ ]:
def get_rnn_embeddings(texts, device):
    """Extracts RAW semantic features using sequential modeling (BiLSTM)."""
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BiLSTMEncoder(vocab_size=tokenizer.vocab_size).to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting RNN (BiLSTM)"):
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)['input_ids'].to(device)
            embeddings.append(model(inputs).squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [ ]:
def get_bert_embeddings(texts, device):
    """Extracts RAW semantic features using bidirectional Auto-Encoder (BERT)."""
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased').to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting BERT"):
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
            embeddings.append(model(**inputs).pooler_output.squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [ ]:
def get_roberta_embeddings(texts, device):
    """Extracts RAW semantic features using optimized Auto-Encoder (RoBERTa)."""
    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    model = RobertaModel.from_pretrained('roberta-base').to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting RoBERTa"):
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
            embeddings.append(model(**inputs).pooler_output.squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [ ]:
def get_gpt2_embeddings(texts, device):
    """Extracts RAW semantic features using auto-regressive Decoder (GPT-2)."""
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2Model.from_pretrained('gpt2').to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting GPT-2"):
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
            # Use the last hidden state of the last token for causal models
            embeddings.append(model(**inputs).last_hidden_state[:, -1, :].squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [ ]:
def get_clip_text_embeddings(texts, device):
    """Extracts RAW semantic features using natively multimodal encoder (CLIP Text)."""
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting CLIP Text"):
            inputs = processor(text=text, return_tensors="pt", padding=True, truncation=True).to(device)
            embeddings.append(model(**inputs).pooler_output.squeeze().cpu().numpy())
    return model, np.array(embeddings)

### Execution

In [ ]:
def execute_and_save(modality, model_name, extraction_func, data, device):
    """Wrapper function to execute extraction, measure hardware metrics, and save data."""
    
    # 1. Start VRAM tracking
    if torch.cuda.is_available(): 
        torch.cuda.reset_peak_memory_stats(device)
        
    start_time = time.time()
    
    # 2. Execute Modular Function
    model_instance, raw_embeddings = extraction_func(data, device)
    
    # 3. Stop Tracking
    exec_time = time.time() - start_time
    peak_vram = torch.cuda.max_memory_allocated(device) / (1024**2) if torch.cuda.is_available() else 0
    
    # 4. Save RAW vector data to disk
    save_path = os.path.join(DATA_DIR, f"X_{modality}_{model_name}_raw_{CURRENT_DATASET}.npy")
    np.save(save_path, raw_embeddings)
    print(f"✔️ Saved: {save_path} | Shape: {raw_embeddings.shape}")
    
    # 5. Strictly enforce memory cleanup
    del model_instance
    del raw_embeddings
    gc.collect()
    if torch.cuda.is_available(): 
        torch.cuda.empty_cache()
        
    # Return metrics for the dataframe
    return {"Modality": modality.capitalize(), "Model": model_name, "Time(s)": round(exec_time, 2), "Peak VRAM(MB)": round(peak_vram, 2)}

# Run

In [ ]:
green_metrics = []

# Define mapping of model names to their specific functions
vision_pipeline = {
    "resnet50": get_resnet50_embeddings,
    "mobilenet_v3": get_mobilenet_v3_embeddings,
    "vit": get_vit_embeddings,
    "deit": get_deit_embeddings,
    "clip_vision": get_clip_vision_embeddings
}

text_pipeline = {
    "rnn": get_rnn_embeddings,
    "bert": get_bert_embeddings,
    "roberta": get_roberta_embeddings,
    "gpt2": get_gpt2_embeddings,
    "clip_text": get_clip_text_embeddings
}

print("\n" + "="*40 + "\nINITIATING VISION PIPELINE\n" + "="*40)
for name, func in vision_pipeline.items():
    metrics = execute_and_save("vision", name, func, image_paths, device)
    green_metrics.append(metrics)
    
print("\n" + "="*40 + "\nINITIATING TEXT PIPELINE\n" + "="*40)
for name, func in text_pipeline.items():
    metrics = execute_and_save("text", name, func, captions, device)
    green_metrics.append(metrics)
    
# Save the consolidated Green AI metrics
df_green = pd.DataFrame(green_metrics)
df_green.to_pickle(os.path.join(DATA_DIR, f'unimodal_green_metrics_RAW_{CURRENT_DATASET}.pkl'))

print("\n Green Metrics" )
display(df_green)